# ML в бизнесе: ДЗ 3 "Детекция аномалий и антифрод"

Задание разбито на две обязательные части и две опциональные.

## Система оценивания домашнего задания

Максимально можно получить **10 баллов** + **2 доп. балла** за каждую опциональную часть. Минимум из полученного суммарного балла и 10 считается оценкой за задание по десятибалльной шкале.

| Часть | Баллы |
|---|---|
| Часть 1. Табличные детекторы аномалий | 4 |
| Часть 2. Транзакционные эмбеддинги vs классика | 6 |
| Часть 3. Граф-подход (опционально) | +2 |
| Часть 4. PYOD vs PYTOD: производительность (опционально) | +2 |

---

## Часть 1. Табличные детекторы аномалий

**Цель:** сравнить несколько unsupervised детекторов аномалий на реальных данных и убедиться, что ROC-AUC — плохая метрика для сильно несбалансированных датасетов.

**Датасет:** `creditcard.zip` из репозитория https://github.com/jeffprosise/Machine-Learning/blob/master/Data/creditcard.zip  
(~285k транзакций, ~0.17% fraud, 30 признаков — PCA-компоненты + Amount + Time)

---

**[2 балла] Задание 1.1 — сравнение детекторов**

Выберите **не менее 4 детекторов** из `pyod` (например, `IForest`, `LOF`, `HBOS` и один нейросетевой — `AutoEncoder` или `VAE`) и сравните их на всей выборке без меток. Выставьте `contamination` равным реальной доле fraud. Посчитайте AP и ROC-AUC для каждого, постройте PR-кривые на одном графике.

**[2 балла] Вопрос 1.2:** Вы получили ROC-AUC = 0.95 и Average Precision = 0.12. Какой показатель точнее отражает качество модели на данной задаче? Выберите вариант и обоснуйте в 1-2 предложениях.
- А) ROC-AUC, он нечувствителен к балансу классов
- Б) Average Precision, она учитывает дисбаланс классов
- В) Оба одинаково информативны при таком дисбалансе

*Ваш ответ:*

In [1]:
!pip install pyod -q

In [2]:
!pip install matplotlib
!pip install numpy

In [5]:
!pip install scikit-learn

In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve,  PrecisionRecallDisplay
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Загрузка датасета
url = "https://github.com/jeffprosise/Machine-Learning/raw/master/Data/creditcard.zip"
df = pd.read_csv(url)

X = df.drop("Class", axis=1).values
y = df["Class"].values

contamination = y.mean()
print(f"Доля fraud: {contamination:.4f}")

# --- ВАШ КОД ---
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.hbos import HBOS
# from pyod.models.auto_encoder import AutoEncoder

contamination = y.mean()
print(f"Размер выборки: {X.shape}")
print(f"Доля fraud (contamination): {contamination:.4f}")
print(f"Положительных примеров: {y.sum()} из {len(y)}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

detectors = {
    "IForest": IForest(contamination=contamination, random_state=42, n_jobs=-1),
    "LOF": LOF(contamination=contamination, n_neighbors=20, n_jobs=-1),
    "HBOS": HBOS(contamination=contamination),
    "AutoEncoder": AutoEncoder(
        contamination=contamination,
        hidden_neuron_list=[32, 16, 32],   # bottleneck=16
        epoch_num=20,
        batch_size=256,
        verbose=0,
    ),
}

results = []
pr_curves = {}
for name, model in detectors.items():
    print(f"\n--- {name} ---")
    model.fit(X_scaled)
    scores = model.decision_scores_

    roc = roc_auc_score(y, scores)
    ap = average_precision_score(y, scores)
    precision, recall, _ = precision_recall_curve(y, scores)

    results.append({"Detector": name, "ROC-AUC": roc, "AP": ap})
    pr_curves[name] = (recall, precision, ap)
    print(f"ROC-AUC = {roc:.4f} | AP = {ap:.4f}")

results_df = pd.DataFrame(results).sort_values("AP", ascending=False).reset_index(drop=True)
print("\n=== Итоговое сравнение ===")
print(results_df.to_string(index=False))

plt.figure(figsize=(8, 6))
baseline = y.mean()  # точность случайного классификатора = доля положительного класса
for name, (recall, precision, ap) in pr_curves.items():
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})", linewidth=2)

plt.axhline(baseline, color="gray", linestyle="--",
            label=f"Random (AP={baseline:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR-кривые детекторов аномалий (creditcard)")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

# Ответ на задание 1.2
Average Precision, так как она учитывает дисбаланс классов.

---

## Часть 2. Транзакционные эмбеддинги vs классика

**Цель:** сравнить три подхода к детекции фрода на транзакционных данных:
1. Ручные фичи (агрегаты по клиенту) + LightGBM (supervised)
2. COLES-эмбеддинги (self-supervised) + LightGBM
3. VAE из `pyod` (unsupervised)

**Датасет:** BankSim — https://github.com/atavci/fraud-detection-on-banksim-data/blob/master/Data/synthetic-data-from-a-financial-payment-system/bs140513_032310.csv  
(~594k транзакций физических лиц, ~1.2% fraud, колонки: `step`, `customer`, `age`, `gender`, `zipcodeOri`, `merchant`, `zipMerchant`, `category`, `amount`, `fraud`)

**Сплит:** train/val **по клиентам** (не по строкам), стратифицированный по `has_fraud`.  
Используйте `test_size=0.1, random_state=42`.

**Метрика везде:** `average_precision_score` на val выборке.

---

**[2 балла] Задание 2.1 — Подход 1: ручные фичи + LightGBM (supervised)**

Агрегируйте транзакции в признаки на уровне клиента — число транзакций, статистики по `amount`, число уникальных мерчантов и категорий (и всё, что сочтёте полезным). Добавьте `fraud_rate` (долю фродовых транзакций у клиента) — **только в train**, в val это будет таргет-утечка.

Таргет: `has_fraud` на уровне клиента. Обучите `LGBMClassifier` на train, предскажите вероятность на val. Выведите `average_precision_score`.

**[2 балла] Задание 2.2 — Подход 2: COLES + LightGBM (self-supervised)**

Используйте `pytorch-lifestream` (PTLS): обучите `CoLESModule` на train-транзакциях без меток, получите эмбеддинги для клиентов из train и val, затем обучите `LGBMClassifier` на эмбеддингах и выведите `average_precision_score`. За основу возьмите код из семинара 3.

**[1 балл] Задание 2.3 — Подход 3: VAE (unsupervised)**

Обучите `VAE` из `pyod` на train-транзакциях без меток (категориальные признаки закодируйте числами). Скор аномальности по транзакциям агрегируйте на уровень клиента — агрегацию выберите самостоятельно и обоснуйте в 1 предложении. Выведите `average_precision_score` на val.

*Ваше обоснование агрегации:*

In [ ]:
!pip install -q pytorch-lifestream lightgbm pytorch_lightning==1.9.0 pyod

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

# Загрузка BankSim
url = "https://raw.githubusercontent.com/atavci/fraud-detection-on-banksim-data/master/Data/synthetic-data-from-a-financial-payment-system/bs140513_032310.csv"
data = pd.read_csv(url)

# zipcodeOri и zipMerchant принимают одно уникальное значение — убираем
data = data.drop(["zipMerchant", "zipcodeOri"], axis=1)

# Сплит по клиентам
grouped = data.groupby("customer").agg(
    num=("fraud", "count"),
    fraud=("fraud", "sum")
)
grouped["has_fraud"] = grouped["fraud"] > 0

train_customers, val_customers = train_test_split(
    grouped.index, test_size=0.1, random_state=42, stratify=grouped.has_fraud
)

train_df = data[data["customer"].isin(train_customers)]
val_df   = data[data["customer"].isin(val_customers)]

print(f"Train fraud transactions: {train_df['fraud'].sum()}")
print(f"Val fraud transactions:   {val_df['fraud'].sum()}")

# --- ВАШ КОД для Заданий 2.1, 2.2, 2.3 ---

# 2.1




#2.2



#2.3

**[1 балл] Задание 2.4 — сравнительная таблица**

Сведите результаты трёх подходов в таблицу и прокомментируйте разницу в 2-3 предложениях.

| Подход | Average Precision | Комментарий |
|---|---|---|
| Ручные фичи + LGBM | | |
| COLES + LGBM | | |
| VAE (unsupervised) | | |

*Ваш комментарий:*

---

## Часть 3. Граф-подход на реальных данных (опционально)

**Цель:** применить граф-алгоритм детекции аномалий на датасете, где транзакция является вершиной графа.

**[+2 балла] Задание:**

Возьмите датасет **Elliptic Bitcoin Dataset**: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set

Датасет описывает граф биткоин-транзакций: вершины — транзакции, рёбра — денежные потоки между ними. Метки: `licit` / `illicit` / `unknown`.

1. Загрузите граф из `elliptic_txs_edgelist.csv` и признаки из `elliptic_txs_features.csv`.
2. Постройте объект `torch_geometric.data.Data` (используйте только вершины с известными метками).
3. Примените один из алгоритмов PyGOD — `DOMINANT` или `CONAD`.
4. Посчитайте `roc_auc_score` и `average_precision_score`. Если используете `DOMINANT` — попробуйте разделить структурный и контекстный скоры и сравните их по отдельности.
5. Сравните с бейзлайном: `IForest` из `pyod` только по табличным признакам вершин (без рёбер).

Ответьте в 3-5 предложениях: даёт ли граф-информация прирост качества на этом датасете?

*Ваш ответ:*

In [ ]:
!pip install torch_geometric pygod pyod -q

import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from pygod.detector import DOMINANT
from sklearn.metrics import roc_auc_score, average_precision_score

edges = pd.read_csv("elliptic_txs_edgelist.csv", header=None)
features = pd.read_csv("elliptic_txs_features.csv", header=None)

# --- ВАШ КОД ---

---

## Часть 4. PYOD vs PYTOD: сравнение производительности (опционально)

**Цель:** убедиться, что GPU-ускоренный `pytod` даёт реальный прирост скорости на тех же данных.

**[+2 балла] Задание:**

Используйте датасет creditcard из Части 1. Выберите **не менее 5 алгоритмов**, реализованных в обеих библиотеках (например, `IForest`, `LOF`, `HBOS`, `CBLOF`, `OCSVM`). Для каждого замерьте время fit в `pyod` и `pytod`. Постройте сравнительный bar-chart: ось X — алгоритм, ось Y — время (сек), два столбца (CPU vs GPU).

Прокомментируйте в 2-3 предложениях: на каких алгоритмах ускорение наиболее заметно и почему?

*Ваш комментарий:*

In [ ]:
!pip install pytod pyod -q

import time
import numpy as np
import matplotlib.pyplot as plt

# X, y, contamination уже загружены в Части 1

results = {}  # {("AlgoName", "library"): seconds}

# Пример:
# from pyod.models.iforest import IForest
# from pytod.models.iforest import IForest as IForestGPU
#
# t0 = time.time()
# IForest(contamination=contamination).fit(X)
# results[("IForest", "pyod")] = time.time() - t0

# --- ВАШ КОД ---